# Clase 053 — Validación temporal: TimeSeriesSplit + walk-forward

Serie sintética (trend + seasonality + noise). Comparamos KFold (leak) vs TimeSeriesSplit vs walk-forward expanding/rolling.

Requiere: `numpy`, `pandas`, `scikit-learn`, `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error

rng = np.random.default_rng(42)
np.random.seed(42)

n = 1000
t = np.arange(n)
trend = 0.02 * t
season = 3 * np.sin(2 * np.pi * t / 50)
noise = rng.normal(0, 0.5, n)
y = trend + season + noise
print('serie:', y.shape, 'min', y.min().round(2), 'max', y.max().round(2))

## 🧠 Intuición previa

Validar una serie temporal es como preparar un examen: **no podés estudiar con las respuestas del futuro**. Si mezclás las fechas al azar (como hace `KFold` con `shuffle`), el modelo entrena viendo días que en producción todavía no ocurrieron, y sacás una nota inflada que jamás vas a repetir en vivo. La validación temporal (`TimeSeriesSplit`, walk-forward) respeta la flecha del tiempo: entrenás solo con el pasado y evaluás sobre el futuro, exactamente como pasa en la realidad.

## 1. Features con lags

Tres lags + rolling mean. Importante: imputar al inicio (NaN por shift).

In [ ]:
df = pd.DataFrame({'y': y})
for lag in (1, 7, 30):
    df[f'lag_{lag}'] = df['y'].shift(lag)
df['roll_mean_7'] = df['y'].shift(1).rolling(7).mean()
df = df.dropna().reset_index(drop=True)
X = df.drop(columns='y').values
y_lag = df['y'].values
print('X', X.shape, 'y', y_lag.shape)

## 2. Leakage de KFold (assert)

KFold mezcla pasado y futuro. Verificamos que train tiene índices mayores que test (imposible en producción).

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
leak_count = 0
for tr, te in kf.split(X):
    if tr.max() > te.min():
        leak_count += 1
assert leak_count == 5, 'KFold debería mezclar todos los folds'
print(f'KFold tiene leakage en {leak_count}/5 folds (train usa futuro de test)')

model = Ridge(alpha=1.0, random_state=42)
mae_kf = -cross_val_score(model, X, y_lag, cv=kf, scoring='neg_mean_absolute_error')
print(f'KFold MAE (optimista por leak): {mae_kf.mean():.4f} ± {mae_kf.std():.4f}')

## 3. TimeSeriesSplit (sklearn)

Split secuencial: train siempre antes que test. Sin leakage.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
for i, (tr, te) in enumerate(tscv.split(X)):
    assert tr.max() < te.min(), 'TSSplit no debe tener overlap temporal'
    print(f'fold {i}: train [0..{tr.max()}] ({len(tr)}) -> test [{te.min()}..{te.max()}] ({len(te)})')

mae_ts = -cross_val_score(model, X, y_lag, cv=tscv, scoring='neg_mean_absolute_error')
print(f'\nTSSplit MAE (realista): {mae_ts.mean():.4f} ± {mae_ts.std():.4f}')

## 4. Walk-forward expanding manual

In [ ]:
def walk_forward_validation(X, y, model, initial_train_size, step):
    """Walk-forward expanding: train crece, test avanza `step` puntos."""
    results = []
    n = len(X)
    start = initial_train_size
    while start + step <= n:
        tr = np.arange(0, start)
        te = np.arange(start, start + step)
        model.fit(X[tr], y[tr])
        pred = model.predict(X[te])
        mae = mean_absolute_error(y[te], pred)
        results.append({'fold': len(results), 'train_end': start, 'test_size': step, 'mae': mae})
        start += step
    return pd.DataFrame(results)

res_exp = walk_forward_validation(X, y_lag, Ridge(alpha=1.0, random_state=42),
                                   initial_train_size=200, step=100)
print(res_exp)
print(f'\nExpanding MAE: {res_exp.mae.mean():.4f} ± {res_exp.mae.std():.4f}')

## 5. Walk-forward rolling (ventana fija)

In [ ]:
def rolling_window_validation(X, y, model, window_size, step):
    results = []
    n = len(X)
    start = window_size
    while start + step <= n:
        tr = np.arange(start - window_size, start)
        te = np.arange(start, start + step)
        model.fit(X[tr], y[tr])
        pred = model.predict(X[te])
        mae = mean_absolute_error(y[te], pred)
        results.append({'fold': len(results), 'train_start': tr[0], 'train_end': start, 'mae': mae})
        start += step
    return pd.DataFrame(results)

res_roll = rolling_window_validation(X, y_lag, Ridge(alpha=1.0, random_state=42),
                                      window_size=200, step=100)
print(res_roll)
print(f'\nRolling MAE: {res_roll.mae.mean():.4f} ± {res_roll.mae.std():.4f}')

## 6. Comparativa final

In [ ]:
summary = pd.DataFrame({
    'estrategia': ['KFold (leak)', 'TimeSeriesSplit', 'Expanding manual', 'Rolling manual'],
    'MAE_mean': [mae_kf.mean(), mae_ts.mean(), res_exp.mae.mean(), res_roll.mae.mean()],
    'MAE_std':  [mae_kf.std(),  mae_ts.std(),  res_exp.mae.std(),  res_roll.mae.std()],
}).round(4)
print(summary)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(summary.estrategia, summary.MAE_mean, yerr=summary.MAE_std, color=['#888', '#3a7', '#37a', '#a73'])
ax.set_ylabel('MAE')
ax.set_title('KFold subestima error vs estrategias temporales')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Ejercicios

1. Agregá `gap=7` a `TimeSeriesSplit` y volé a evaluar. ¿Cambia el MAE?
2. Implementá `purged_kfold` (López de Prado) que elimina del train cualquier sample con overlap de target.
3. Cambiá a XGBoost. ¿Qué estrategia gana?

## Conclusiones

- KFold sobre series temporales infla métricas porque entrena con futuro.
- `TimeSeriesSplit` resuelve el caso estándar; agregá `gap` si tus features tienen lags.
- Expanding usa toda la historia; rolling "olvida" — eligí según estabilidad del proceso.

## ✅ Soluciones de los ejercicios

Resolvemos los 5 ejercicios del README con una serie sintética (tendencia + estacionalidad + ruido) y features lag, todo offline y con `n_jobs=1`.

**Ej. 1 — TSSplit vs KFold (leak).** Con la misma serie, `KFold(shuffle=True)` mezcla pasado y futuro e infla el score; `TimeSeriesSplit` respeta el orden y da la estimación realista.

In [ ]:

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score

rng = np.random.default_rng(7)
n = 600
t = np.arange(n)
serie = 0.02 * t + 3 * np.sin(2 * np.pi * t / 50) + rng.normal(0, 0.5, n)
df = pd.DataFrame({"y": serie})
for lag in (1, 7, 30):
    df[f"lag_{lag}"] = df["y"].shift(lag)
df = df.dropna().reset_index(drop=True)
X, ylag = df.drop(columns="y").values, df["y"].values

rmse_kf = -cross_val_score(Ridge(), X, ylag, cv=KFold(5, shuffle=True, random_state=42),
                           scoring="neg_root_mean_squared_error")
rmse_ts = -cross_val_score(Ridge(), X, ylag, cv=TimeSeriesSplit(5),
                           scoring="neg_root_mean_squared_error")
print(f"KFold shuffle   RMSE = {rmse_kf.mean():.4f}  (optimista: entrena con futuro)")
print(f"TimeSeriesSplit RMSE = {rmse_ts.mean():.4f}  (realista: solo pasado)")
assert rmse_kf.mean() <= rmse_ts.mean() + 1e-9, "KFold suele dar error mas bajo (inflado)"

**Ej. 2 — Walk-forward expanding con `TimeSeriesSplit`.** Iteramos los 5 folds y reportamos el RMSE de cada uno; el train crece en cada paso.

In [ ]:

import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)
scores = []
for i, (tr, te) in enumerate(tscv.split(X)):
    m = Ridge().fit(X[tr], ylag[tr])
    rmse = root_mean_squared_error(ylag[te], m.predict(X[te]))
    scores.append(rmse)
    print(f"fold {i}: train [0..{tr.max()}] ({len(tr)} obs) -> test {len(te)} obs | RMSE={rmse:.4f}")
scores = np.array(scores)
print(f"expanding RMSE medio = {scores.mean():.4f}")
assert len(scores) == 5

**Ej. 3 — Rolling window (ventana fija).** `TimeSeriesSplit(max_train_size=100)` simula el walk-forward de ventana fija: el train 'olvida' lo más viejo.

In [ ]:

import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

roll = TimeSeriesSplit(n_splits=5, max_train_size=100)
sizes, rmses = [], []
for tr, te in roll.split(X):
    sizes.append(len(tr))
    m = Ridge().fit(X[tr], ylag[tr])
    rmses.append(root_mean_squared_error(ylag[te], m.predict(X[te])))
print("tamano de train por fold:", sizes, "(fijo <= 100 = ventana rodante)")
print("RMSE por fold:", np.round(rmses, 4), f"| medio {np.mean(rmses):.4f}")
assert max(sizes) <= 100, "la ventana debe quedar acotada a 100"

**Ej. 4 — `gap` para evitar fuga del target-lag.** Con `lag_1` como feature, el primer punto de test comparte información con el último de train. `gap=1` inserta un hueco temporal que lo evita.

In [ ]:

import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

sin_gap = -cross_val_score(Ridge(), X, ylag, cv=TimeSeriesSplit(5, gap=0),
                           scoring="neg_root_mean_squared_error")
con_gap = -cross_val_score(Ridge(), X, ylag, cv=TimeSeriesSplit(5, gap=1),
                           scoring="neg_root_mean_squared_error")
# comprobamos que el gap efectivamente separa train de test
for tr, te in TimeSeriesSplit(5, gap=1).split(X):
    assert te.min() - tr.max() >= 2, "con gap=1 debe haber >=1 punto de separacion"
print(f"RMSE sin gap = {sin_gap.mean():.4f}")
print(f"RMSE gap=1   = {con_gap.mean():.4f}  (mas conservador: test no roza el train)")

**Ej. 5 — Score con dispersión.** Reportar `mean ± std` del RMSE por fold, no solo la media: la dispersión avisa si el error es inestable en el tiempo.

In [ ]:

import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

rmse = -cross_val_score(Ridge(), X, ylag, cv=TimeSeriesSplit(5),
                        scoring="neg_root_mean_squared_error")
print("RMSE por fold:", np.round(rmse, 4))
print(f"RMSE = {rmse.mean():.4f} ± {rmse.std():.4f}")
print("reportar solo la media esconde que algunos periodos son mucho mas dificiles que otros")
assert rmse.std() >= 0